## Author: Gabriel Vance
## Class: DSC630 - Predictive Analytics
## Professor: Professor Andrew Hua
## Date: October 27, 2025

# Purpose: This notebook performs the tasks outlined in 10.2 Assignment: Recommender Systems. 

Using the small MovieLens data set, create a recommender system that allows users to input a movie they like (in the data set) and recommends ten other movies for them to watch. In your write-up, clearly explain the recommender system process and all steps performed. If you are using a method found online, be sure to reference the source.



# Movie Recommender System Implementation

## Overview
This project implements a **collaborative filtering** movie recommender system using the MovieLens dataset. The system uses **item-based collaborative filtering** with **cosine similarity** to recommend movies to users based on their preferences.

## Methodology
The recommender system works by:
1. **Data Loading & Preprocessing**: Loading movie and rating data, creating user-item matrices
2. **Similarity Calculation**: Computing cosine similarity between movies based on user ratings
3. **Recommendation Generation**: Finding movies most similar to a user's input movie
4. **Filtering & Ranking**: Presenting the top 10 most similar movies

## Data Sources
- **Movies Dataset**: Contains movie information (ID, title, genres)
- **Ratings Dataset**: Contains user ratings (userId, movieId, rating, timestamp)

**Reference**: This implementation follows collaborative filtering principles from Ricci, F., Rokach, L., & Shapira, B. (2015). *Recommender Systems Handbook*. Springer.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

In [ ]:
# Step 1: Load the MovieLens dataset
print("Loading MovieLens dataset...")

# Load movies data
movies = pd.read_csv('/workspaces/data_science_classes_windows/fall2025/dsc630_data_prediction/week10/data/movies.csv')
print(f"Movies dataset shape: {movies.shape}")
print(f"Sample movies:")
print(movies.head())

# Load ratings data (using sample.csv as it's a smaller subset)
ratings = pd.read_csv('/workspaces/data_science_classes_windows/fall2025/dsc630_data_prediction/week10/data/sample.csv')
print(f"\nRatings dataset shape: {ratings.shape}")
print(f"Sample ratings:")
print(ratings.head())

print(f"\nUnique users: {ratings['userId'].nunique()}")
print(f"Unique movies: {ratings['movieId'].nunique()}")
print(f"Rating range: {ratings['rating'].min()} - {ratings['rating'].max()}")

In [ ]:
# Step 2: Data preprocessing and exploration
print("Data Preprocessing...")

# Merge movies and ratings data
movie_ratings = pd.merge(ratings, movies, on='movieId', how='inner')
print(f"Combined dataset shape: {movie_ratings.shape}")

# Check for movies with sufficient ratings (at least 10 ratings for better similarity calculation)
movie_rating_counts = movie_ratings.groupby('movieId').size()
popular_movies = movie_rating_counts[movie_rating_counts >= 10].index
print(f"Movies with at least 10 ratings: {len(popular_movies)}")

# Filter to only include popular movies
filtered_ratings = movie_ratings[movie_ratings['movieId'].isin(popular_movies)]
print(f"Filtered dataset shape: {filtered_ratings.shape}")

# Display most rated movies
top_movies = filtered_ratings.groupby('title').size().sort_values(ascending=False).head(10)
print(f"\nTop 10 most rated movies:")
print(top_movies)

In [ ]:
# Step 3: Create User-Item Matrix
print("Creating user-item matrix...")

# Create pivot table (user-item matrix) with ratings
user_item_matrix = filtered_ratings.pivot_table(
    index='userId', 
    columns='movieId', 
    values='rating',
    fill_value=0
)

print(f"User-item matrix shape: {user_item_matrix.shape}")
print(f"Sparsity: {(user_item_matrix == 0).sum().sum() / (user_item_matrix.shape[0] * user_item_matrix.shape[1]):.4f}")

# Create movie-user matrix (transpose for item-based filtering)
movie_user_matrix = user_item_matrix.T
print(f"Movie-user matrix shape: {movie_user_matrix.shape}")

# Display sample of the matrix
print("\nSample of user-item matrix:")
print(user_item_matrix.iloc[:5, :5])

In [ ]:
# Step 4: Calculate Movie Similarity Matrix using Cosine Similarity
print("Calculating movie similarity matrix...")

# Calculate cosine similarity between movies
# We use the movie-user matrix where each row represents a movie and columns are users
movie_similarity = cosine_similarity(movie_user_matrix)

# Convert to DataFrame for easier handling
movie_similarity_df = pd.DataFrame(
    movie_similarity, 
    index=movie_user_matrix.index, 
    columns=movie_user_matrix.index
)

print(f"Movie similarity matrix shape: {movie_similarity_df.shape}")

# Display sample similarities for first movie
sample_movie_id = movie_similarity_df.index[0]
print(f"\nSimilarity scores for movie ID {sample_movie_id}:")
print(movie_similarity_df.loc[sample_movie_id].sort_values(ascending=False).head(6))

In [ ]:
# Step 5: Create Movie Recommendation Function
def get_movie_recommendations(movie_title, n_recommendations=10):
    """
    Recommends movies based on similarity to the input movie.
    
    Parameters:
    movie_title (str): The title of the movie to base recommendations on
    n_recommendations (int): Number of recommendations to return
    
    Returns:
    DataFrame: Top N recommended movies with similarity scores
    """
    
    # Find the movie ID for the given title
    movie_match = movies[movies['title'].str.contains(movie_title, case=False, na=False)]
    
    if movie_match.empty:
        print(f"Movie '{movie_title}' not found in the dataset.")
        print("Available movies (sample):")
        print(movies['title'].head(10).tolist())
        return None
    
    # If multiple matches, take the first one
    if len(movie_match) > 1:
        print(f"Multiple movies found with '{movie_title}'. Using: {movie_match.iloc[0]['title']}")
    
    movie_id = movie_match.iloc[0]['movieId']
    movie_title_full = movie_match.iloc[0]['title']
    
    # Check if movie is in our similarity matrix
    if movie_id not in movie_similarity_df.index:
        print(f"Movie '{movie_title_full}' doesn't have enough ratings for recommendations.")
        return None
    
    # Get similarity scores for this movie
    similarity_scores = movie_similarity_df.loc[movie_id]
    
    # Sort by similarity (excluding the movie itself)
    similar_movies = similarity_scores.drop(movie_id).sort_values(ascending=False)
    
    # Get top N recommendations
    top_recommendations = similar_movies.head(n_recommendations)
    
    # Create recommendations dataframe with movie details
    recommendations = []
    for movie_id_rec, similarity_score in top_recommendations.items():
        movie_info = movies[movies['movieId'] == movie_id_rec].iloc[0]
        recommendations.append({
            'movieId': movie_id_rec,
            'title': movie_info['title'],
            'genres': movie_info['genres'],
            'similarity_score': similarity_score
        })
    
    recommendations_df = pd.DataFrame(recommendations)
    
    print(f"\nTop {n_recommendations} recommendations for '{movie_title_full}':")
    print("=" * 70)
    
    for i, row in recommendations_df.iterrows():
        print(f"{i+1:2d}. {row['title']}")
        print(f"    Genres: {row['genres']}")
        print(f"    Similarity Score: {row['similarity_score']:.4f}")
        print()
    
    return recommendations_df

print("Movie recommendation function created successfully!")

In [ ]:
# Step 6: Test the Recommender System with Popular Movies
print("Testing the Movie Recommender System...")
print("\n" + "="*80)

# Test with a popular movie
test_movie = "Toy Story"
recommendations = get_movie_recommendations(test_movie)

print("\n" + "="*80)

# Test with another popular movie
test_movie2 = "Jurassic Park"
recommendations2 = get_movie_recommendations(test_movie2)

In [ ]:
# Step 7: Interactive Recommender System
def search_movies(search_term):
    """Helper function to search for movies by title"""
    matches = movies[movies['title'].str.contains(search_term, case=False, na=False)]
    if len(matches) > 0:
        print(f"Movies matching '{search_term}':")
        for i, (idx, movie) in enumerate(matches.head(10).iterrows()):
            print(f"{i+1:2d}. {movie['title']} ({movie['genres']})")
    else:
        print(f"No movies found matching '{search_term}'")
    return matches

def interactive_recommender():
    """Interactive movie recommender function"""
    print("🎬 Welcome to the Movie Recommender System!")
    print("=" * 50)
    print("You can:")
    print("1. Search for movies: search_movies('movie name')")
    print("2. Get recommendations: get_movie_recommendations('movie name')")
    print("3. Example: get_movie_recommendations('Star Wars')")
    print("=" * 50)

# Run the interactive setup
interactive_recommender()

In [ ]:
# Step 8: System Analysis and Evaluation
print("System Analysis and Performance Evaluation")
print("=" * 50)

# Analyze the dataset coverage
print(f"Total movies in dataset: {len(movies)}")
print(f"Movies with ratings: {len(movie_user_matrix)}")
print(f"Coverage: {len(movie_user_matrix) / len(movies) * 100:.2f}%")

# Most popular genres analysis
all_genres = []
for genres in movies['genres']:
    if pd.notna(genres):
        all_genres.extend(genres.split('|'))

genre_counts = pd.Series(all_genres).value_counts().head(10)
print(f"\nTop 10 Genres in Dataset:")
print(genre_counts)

# Test with different movie genres
print(f"\n" + "="*60)
print("Testing recommendations across different genres:")

# Horror movie test
horror_test = "Halloween"
print(f"\n--- Testing Horror Movie: {horror_test} ---")
horror_recs = get_movie_recommendations(horror_test)

# Comedy movie test  
comedy_test = "Dumb and Dumber"
print(f"\n--- Testing Comedy Movie: {comedy_test} ---")
comedy_recs = get_movie_recommendations(comedy_test)